In [1]:
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message="invalid value encountered in divide")

LEVEL = 0.95

In [2]:
# Load data and settings from the bootstrap
BOOT_FILE = "epa_bootstrap_tables.npz"  
saved = np.load(BOOT_FILE)

# Settings
MAX_YTG = saved["MAX_YTG"].item()
TD_POINTS = saved["TD_POINTS"].item()
FG_POINTS = saved["FG_POINTS"].item()
MIN_TIME_TO_SCORE = saved["MIN_TIME_TO_SCORE"].item()
END_OF_HALF_SECS = saved["END_OF_HALF_SECS"].item()
TABLE_NAMES = saved["TABLE_NAMES"].tolist()

# The tables with estimates from epa_calc
point_tables = {k: saved[f"point_{k}"] for k in TABLE_NAMES}
EP1, P_CONVERT, SUCCESS, TD_SHARE, PUNT, P_FG, FG_EP = (point_tables[k] for k in TABLE_NAMES)

# Tables with esitmates from bootstarp resamples
stacked = {k: saved[f"boot_{k}"] for k in TABLE_NAMES}
boot_tables = [{k: stacked[k][b] for k in TABLE_NAMES} for b in range(len(stacked["EP1"]))]
print(f"Loaded {len(boot_tables):,} resamples from {BOOT_FILE}")

# load evaluate_frame()
%run -i evaluate_frame.py

Loaded 1,000 resamples from epa_bootstrap_tables.npz


## Evaluate a situation

In [3]:
# ---- Situation ----
qtr = 3                 # 1-4, or 5 for overtime
clock = "7:00"          # time left in the quarter
yardline_100 = 35       # yards from the opponent's end zone
ydstogo = 4             # yards to go
score_diff = 0          # score difference - negative means trailing
timeouts = 3            # timeouts left

# Evaluate
m, sec = clock.split(":")
secs_in_qtr = int(m) * 60 + int(sec)
game_secs = (4 - qtr) * 900 + secs_in_qtr if qtr <= 4 else secs_in_qtr
row = pd.DataFrame([{
    "qtr": qtr,
    "game_seconds_remaining": game_secs,
    "yardline_100": yardline_100,
    "ydstogo": ydstogo,
    "score_differential": score_diff,
    "posteam_timeouts_remaining": timeouts,
}])

# The estimate using epa_calc
est = evaluate_frame(row).iloc[0]

# Same situation from the bootstrap tables
draws = []
try:
    for T in boot_tables:
        EP1, P_CONVERT, SUCCESS, TD_SHARE, PUNT, P_FG, FG_EP = (T[k] for k in TABLE_NAMES)
        draws.append(evaluate_frame(row))
finally:
    EP1, P_CONVERT, SUCCESS, TD_SHARE, PUNT, P_FG, FG_EP = (point_tables[k] for k in TABLE_NAMES)
draws = pd.concat(draws, ignore_index=True)

num_cols = ["p_convert", "points_if_convert", "points_if_fail", "breakeven_rate",
            "ep_go", "ep_punt", "ep_fg"]
lo = draws[num_cols].quantile((1 - LEVEL) / 2)
hi = draws[num_cols].quantile((1 + LEVEL) / 2)

fmt = lambda v: "n/a" if pd.isna(v) else f"{v:+.2f}"
pct = lambda v: "n/a" if pd.isna(v) else f"{v:.0%}"
show = lambda col, f: f(est[col]) if pd.isna(est[col]) else f"{f(est[col])}  [{f(lo[col])}, {f(hi[col])}]"

rec = est["recommendation"]
ep_cols = {"go": "ep_go", "punt": "ep_punt", "field_goal": "ep_fg"}
others = {o: est[c] for o, c in ep_cols.items() if o != rec and not pd.isna(est[c])}
runner_up = max(others, key=others.get) if others else None
if runner_up:
    margin = draws[ep_cols[rec]] - draws[ep_cols[runner_up]]
    rec_line = (f"{rec.upper()} by {fmt(est[ep_cols[rec]] - est[ep_cols[runner_up]])} over {runner_up}"
                f"  [{fmt(margin.quantile((1 - LEVEL) / 2))}, {fmt(margin.quantile((1 + LEVEL) / 2))}]")
else:
    rec_line = rec.upper()
first = draws["recommendation"].value_counts(normalize=True)

In [4]:
# ---- Print results ----

spot = (f"opp {yardline_100}" if yardline_100 < 50
        else "midfield" if yardline_100 == 50 else f"own {100 - yardline_100}")
score = "tied" if score_diff == 0 else (f"up {score_diff}" if score_diff > 0 else f"down {-score_diff}")
q = "OT" if qtr == 5 else f"Q{qtr}"

print(f"{q} {clock} | 4th & {ydstogo} at {spot} | {score} | {timeouts} TO"
      f"   ({LEVEL:.0%} intervals from {len(boot_tables):,} resamples)")
print(f"  P(convert):                {show('p_convert', pct)}")
print(f"  Points added if you make:  {show('points_if_convert', fmt)}  (vs best kick/punt)")
print(f"  Points lost if you miss:   {show('points_if_fail', fmt)}")
print(f"  Break-even conversion:     {show('breakeven_rate', pct)}")
print(f"  EP go:                     {show('ep_go', fmt)}")
print(f"  EP punt:                   {show('ep_punt', fmt)}")
print(f"  EP field goal:             {show('ep_fg', fmt)}")
print(f"  Recommendation:            {rec_line}"
      + ("  (can't get the ball back: only go/FG-to-tie-or-win count)" if est["desperation"] else "")
      + ("  (last play of the half: only points on this play count)" if est["last_play"] else ""))
print(f"  Ranked first in:           go {first.get('go', 0):.0%} | punt {first.get('punt', 0):.0%}"
      f" | field goal {first.get('field_goal', 0):.0%} of resamples")

Q3 7:00 | 4th & 4 at opp 35 | tied | 3 TO   (95% intervals from 1,000 resamples)
  P(convert):                48%  [46%, 50%]
  Points added if you make:  +2.63  [+2.52, +2.81]  (vs best kick/punt)
  Points lost if you miss:   -3.32  [-3.42, -3.14]
  Break-even conversion:     56%  [53%, 57%]
  EP go:                     +1.22  [+1.11, +1.31]
  EP punt:                   n/a
  EP field goal:             +1.67  [+1.50, +1.77]
  Recommendation:            FIELD_GOAL by +0.45 over go  [+0.25, +0.59]
  Ranked first in:           go 0% | punt 0% | field goal 100% of resamples
